# 00 · Setup

**Purpose**
- Connect Colab to Google Drive
- Clone the GitHub repo into Drive (first run) or update it
- Install packages and check the GPU
- Test the error log and the GitHub push

**Run once per Colab session.** Later notebooks start with the short setup cell in Step 4.

## 1 · Mount Google Drive
- The repo and all data live in Drive, so nothing is lost when the Colab session ends.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Clone or update the repo
- Fill in `GITHUB_USER` once.
- The token comes from **Colab secrets** (🔑 icon in the left sidebar), key `GITHUB_TOKEN`.
- The token is used for cloning only and then removed from the remote URL, so it is never saved in Drive.

In [4]:
import os, subprocess
from google.colab import userdata

GITHUB_USER = "dedeepyakm"   # <- edit once
REPO        = "cw-mpvr"
REPO_DIR    = f"/content/drive/MyDrive/{REPO}"

token = userdata.get("GITHUB_TOKEN")
clean_url = f"https://github.com/{GITHUB_USER}/{REPO}.git"

if not os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clean_url], check=True)
    print("Cloned into", REPO_DIR)
else:
    print("Repo already in Drive:", REPO_DIR)

%cd {REPO_DIR}
!git config user.name  "Dedeepya Korukonda"
!git config user.email "dedeepyakm@gmail.com"   # GitHub > Settings > Emails
!git log --oneline -3

Repo already in Drive: /content/drive/MyDrive/cw-mpvr
/content/drive/MyDrive/cw-mpvr
691808e (HEAD -> main) Run setup notebook on Colab
a01d92b (origin/main, origin/HEAD) Update 00_setup.ipynb
d2e70d8 Add files via upload


## 3 · Install packages
- PyTorch is already on Colab; this adds the rest from `requirements.txt`.

In [5]:
!pip install -q -r requirements.txt

## 4 · Standard setup cell
- **Copy this cell to the top of every notebook** (after mounting Drive).
- Makes `src/` importable, loads the config, fixes the random seed, reports the GPU.

In [6]:
import sys
REPO_DIR = "/content/drive/MyDrive/cw-mpvr"
%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# autoreload removed: broken on Colab's Python 3.13 (see ERROR_LOG E01)
# after editing a file in src/: Runtime > Restart session, then rerun this cell

from src.utils import load_config, set_seed, gpu_info
from src.errorlog import log_error
from src.gitsync import push, pull

cfg = load_config()
set_seed(cfg["seed"])
print(gpu_info())

/content/drive/MyDrive/cw-mpvr
NVIDIA L4 | 23.7 GB | torch 2.11.0+cu130


In [7]:
log_error(
    where="00_setup / step 4 (standard setup cell)",
    error="ModuleNotFoundError: No module named 'imp'",
    cause="Colab moved to Python 3.13; its bundled IPython autoreload extension still imports `imp`, which was removed in Python 3.12",
    fix="Removed %load_ext autoreload; after editing src/ files, restart the session and rerun the setup cell",
    lesson="Check the runtime's Python version first; prefer explicit restarts over magic extensions",
)

Logged as E02:
### E02 (2026-10-25)
- **Where**: 00_setup / step 4 (standard setup cell)
- **Error**: `ModuleNotFoundError: No module named 'imp'`
- **Cause**: Colab moved to Python 3.13; its bundled IPython autoreload extension still imports `imp`, which was removed in Python 3.12
- **Fix**: Removed %load_ext autoreload; after editing src/ files, restart the session and rerun the setup cell
- **Lesson**: Check the runtime's Python version first; prefer explicit restarts over magic extensions



## 5 · Check the folders
- `cw-mpvr/` holds code (in git).
- `cw-mpvr-data/` holds GQA files, images and cached model outputs (**not** in git, too large).

In [8]:
for k, v in cfg["paths"].items():
    print(f"{k:6s} -> {v}  (exists: {os.path.isdir(v)})")

repo   -> /content/drive/MyDrive/cw-mpvr  (exists: True)
data   -> /content/drive/MyDrive/cw-mpvr-data  (exists: True)
cache  -> /content/drive/MyDrive/cw-mpvr-data/cache  (exists: True)


## 6 · How to log an error
- When a cell fails: fix it, then run `log_error(...)` in a new cell.
- The error message is captured automatically from the failed cell.
- Everything goes to `ERROR_LOG.md`, which becomes the "Errors and fixes" section of the report.

Example (commented out so it does not create a fake entry):

In [9]:
# log_error(
#     where="00_setup / step 3",
#     cause="Package version clash between transformers and accelerate",
#     fix="Pinned accelerate>=0.33 in requirements.txt",
#     lesson="Pin versions once the pipeline works",
# )

## 7 · Save work to GitHub
- Save the notebook first (**Ctrl+S**), then push.
- Use short commit messages that say what changed.

In [10]:
push("Run setup notebook on Colab")

M  ERROR_LOG.md
Pushed: Run setup notebook on Colab


In [11]:
%%writefile -a /content/drive/MyDrive/cw-mpvr/src/gitsync.py


def save_notebook(repo_path: str) -> None:
    """Write the notebook currently open in Colab to repo_path (e.g. 'notebooks/01_data_eda.ipynb')."""
    import json
    from google.colab import _message
    nb = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    out = REPO_ROOT / repo_path
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w") as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
    print(f"Saved notebook -> {repo_path}")


def save_and_push(repo_path: str, message: str) -> None:
    """Save the open notebook into the repo, then commit and push everything."""
    save_notebook(repo_path)
    push(message)

Appending to /content/drive/MyDrive/cw-mpvr/src/gitsync.py


In [ ]:
import importlib, src.gitsync
importlib.reload(src.gitsync)
from src.gitsync import save_and_push

save_and_push("notebooks/00_setup.ipynb", "Fix setup cell for Python 3.13; add save_and_push helper")